# Query Translation



In [2]:
import os
import bs4
from dotenv import load_dotenv

from langchain_classic.retrievers import MultiQueryRetriever
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tracers import LangChainTracer
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

C:\Users\amroosha\AppData\Local\Temp\ipykernel_25500\4135347378.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


In [3]:
load_dotenv()

# Setup explicit tracer instance
tracer = LangChainTracer(project_name="multi-query-rag")

#### INDEXING ####

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

#### MULTI-QUERY RETRIEVER ####

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# MultiQueryRetriever uses the LLM to generate multiple variations of the input query
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(),
    llm=llm
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
retriever = vectorstore.as_retriever()

## Rewriting (Multi-Query)

In [ ]:

#### RAG CHAIN ####

prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": multi_query_retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

#### INVOCATION WITH TRACER ####

query = "What is Task Decomposition?"

# Pass the tracer explicitly in the config dict inside invoke()
response = rag_chain.invoke(
    query,
    config={"callbacks": [tracer]}
)

print(response)


--- Response ---
**Task Decomposition** is the process of breaking a complex task into a set of smaller, manageable sub‑tasks or steps.  
- It allows an agent (or an LLM) to understand what needs to be done and to plan ahead.  
- In practice it can be achieved by:  
  1. **Simple prompting** (e.g., “Steps for XYZ.” or “What are the subgoals for achieving XYZ?”).  
  2. **Task‑specific instructions** (e.g., “Write a story outline.”).  
  3. **Human input** (manual guidance or feedback).  

This decomposition is a key component of planning, enabling techniques like Chain‑of‑Thought or Tree‑of‑Thought to reason step‑by‑step or explore multiple reasoning paths.


## Rewriting (RAG Fusion)

In [5]:
query_generation_prompt = ChatPromptTemplate.from_template(
    """You are an AI language model assistant. Your task is to generate 5
different versions of the given user question to retrieve relevant documents from a vector database.
By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines.

Original question: {question}"""
)

# Output parser to convert generated questions into a Python list
generate_queries = (
    query_generation_prompt
    | llm
    | StrOutputParser()
    | (lambda x: [q.strip() for q in x.split("\n") if q.strip()])
)

In [ ]:
def reciprocal_rank_fusion(results: list[list], k: int = 10):
    """
    Reciprocal Rank Fusion (RRF) algorithm to score and re-rank documents.
    results: List of Document lists (one list per generated query)
    k: Smoothing constant (default 60 as per RRF paper)
    """
    fused_scores = {}

    for docs in results:
        for rank, doc in enumerate(docs):
            # Convert document content to string key for hashing
            doc_str = doc.page_content
            if doc_str not in fused_scores:
                fused_scores[doc_str] = {"doc": doc, "score": 0.0}

            # Add RRF score: 1 / (k + rank)
            fused_scores[doc_str]["score"] += 1.0 / (k + (rank + 1))

    # Sort documents by accumulated RRF score in descending order
    reranked_results = sorted(
        fused_scores.values(), key=lambda x: x["score"], reverse=True
    )

    # Return top re-ranked Document objects
    return [item["doc"] for item in reranked_results]


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [7]:
retriever = vectorstore.as_retriever()

retrieval_chain = generate_queries | retriever.map() | reciprocal_rank_fusion

# Final RAG Prompt
rag_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the following context:
{context}

Question: {question}
"""
)

# Complete RAG-Fusion Chain
rag_fusion_chain = (
    {
        "context": retrieval_chain | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)


query = "What is Task Decomposition for LLM agents?"

# Invoke passing tracer explicitly in config
response = rag_fusion_chain.invoke(query, config={"callbacks": [tracer]})

print(response)

**Task decomposition for LLM agents** is the process of breaking a large, complex task into a set of smaller, manageable sub‑goals or steps that the agent can execute sequentially or in parallel. It is a core part of the agent’s planning module and enables the LLM to handle long‑horizon or multi‑step problems efficiently.

Key points from the context:

| Aspect | How it’s done |
|--------|---------------|
| **LLM‑driven prompting** | Simple prompts such as “Steps for XYZ.” or “What are the subgoals for achieving XYZ?” let the LLM generate a list of sub‑tasks. |
| **Task‑specific instructions** | For certain domains, a single instruction can produce a structured outline (e.g., “Write a story outline.” for novel writing). |
| **Human input** | A user can supply or refine the decomposition manually. |
| **External planners** | In some setups, the LLM can translate a problem into PDDL, call a classical planner, and then translate the resulting plan back into natural language. |

The decomp

## Less Abstraction (Decomposition) (Least-to-Most)

In [4]:
# Decomposition
template = """You are a helpful assistant that generates multiple sub-questions related to an input question. \n
The goal is to break down the input into a set of sub-problems / sub-questions that can be answers in isolation. \n
Generate multiple search queries related to: {question} \n
Output (3 queries):"""
prompt_decomposition = ChatPromptTemplate.from_template(template)

In [5]:
# LLM
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# Chain
generate_queries_decomposition = ( prompt_decomposition | llm | StrOutputParser() | (lambda x: x.split("\n")))

# Run
question = "What are the main components of an LLM-powered autonomous agent system?"
questions = generate_queries_decomposition.invoke(question, config={"callbacks": [tracer]})
questions

['1. “What are the core architectural components of an LLM‑powered autonomous agent system?”  ',
 '2. “How do perception, planning, and execution modules integrate with a large language model in an autonomous agent?”  ',
 '3. “What role does memory and knowledge base play in the design of an LLM‑driven autonomous agent?”']

In [6]:
# Prompt
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question: 

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

decomposition_prompt = ChatPromptTemplate.from_template(template)


In [9]:
from operator import itemgetter

def format_qa_pair(question, answer):
    """Format Q and A pair"""
    
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# llm
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

q_a_pairs = ""
for q in questions:
    
    rag_chain = (
    {"context": itemgetter("question") | retriever, 
     "question": itemgetter("question"),
     "q_a_pairs": itemgetter("q_a_pairs")} 
    | decomposition_prompt
    | llm
    | StrOutputParser())

    answer = rag_chain.invoke({"question":q,"q_a_pairs":q_a_pairs}, config={"callbacks": [tracer]})
    q_a_pair = format_qa_pair(q,answer)
    q_a_pairs = q_a_pairs + "\n---\n"+  q_a_pair

In [11]:
print(q_a_pairs)


---
Question: 1. “What are the core architectural components of an LLM‑powered autonomous agent system?”  
Answer: **Core architectural components of an LLM‑powered autonomous agent**

| # | Component | Typical responsibilities | Key references |
|---|-----------|--------------------------|----------------|
| 1 | **LLM controller (brain)** | Generates plans, actions, and natural‑language explanations; acts as the decision‑making core. | Weng 2023, “LLM‑Powered Autonomous Agents” |
| 2 | **Memory subsystem** | • **Short‑term (working) memory** – holds the current context, recent actions, and intermediate reasoning. <br>• **Long‑term memory** – stores facts, past experiences, and learned knowledge (e.g., embeddings, vector store). | Weng 2023 (Memory section) |
| 3 | **Planning & decomposition engine** | Breaks a high‑level goal into sub‑goals, sequences, or “tree‑of‑thoughts”; may use chain‑of‑thought or tree‑of‑thought prompting. | Wei 2022, Yao 2023 |
| 4 | **Reflection / self‑critiq

## More Abstraction (Step-back prompting)

In [ ]:
# Few Shot Examples
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate
examples = [
    {
        "input": "Could the members of The Police perform lawful arrests?",
        "output": "what can the members of The Police do?",
    },
    {
        "input": "Jan Sindel's was born in what country?",
        "output": "what is Jan Sindel's personal history?",
    },
]

# We now transform these to example messages
example_prompt = ChatPromptTemplate.from_messages(
    [
        ("user", "{input}"),
        ("ai", "{output}"),
    ]
)

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """You are an expert at world knowledge. Your task is to step back and paraphrase a question to a more generic step-back question, which is easier to answer. Here are a few examples:""",
        ),
        # Few shot examples
        few_shot_prompt,
        # New question
        ("user", "{question}"),
    ]
)